# 最終課題

In [112]:
# Webスクレイピングに最低限必要なライブラリをインポート
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
from collections import deque # 高速に先頭からデータを取り出す(pop)ために使います
from urllib.parse import urlparse

In [113]:
# アクセスしたいWebサイトのURLを指定
url = 'https://www.musashino-u.ac.jp'

# WebサーバーにHTTPリクエストを送信
# レスポンスを変数に格納しておく
res = requests.get(url)

print(f"ステータスコード:{res.status_code}") # ステータスコードを表示


ステータスコード:200


In [114]:
page_data = {}
visited_url = set()
visit_url_queue = deque([url])
base_domain = urlparse(url).netloc

EXCLUDED_EXTENSIONS = (
    '.pdf', '.jpg', '.jpeg', '.png', '.gif', '.svg', # 画像・PDF
    '.zip', '.rar', '.gz', # 圧縮ファイル
    '.mp4', '.mov', '.wmv', # 動画
    '.mp3', '.wav', # 音声
    '.xlsx', '.xls', '.docx', '.doc', '.pptx', '.ppt' # オフィス文書
)


while visit_url_queue:
    current_url = visit_url_queue.popleft()
    if current_url in visited_url:
        continue

    visited_url.add(current_url)

    # ★修正箇所★
    # どのページを見ているか表示（件数を追加）
    print(f"[{len(visited_url)}件目] Accessing: {current_url}")

    try:
        res = requests.get(current_url)
        res.encoding = res.apparent_encoding # 文字化け対策
        time.sleep(0.5)  # サーバーに負荷をかけないように0.5秒待機

        content_type = res.headers.get('Content-Type', '')
        if 'text/html' not in content_type:
            print(f"  -> スキップ (Content-Type: {content_type})")
            page_data[current_url] = f"HTMLではない ({content_type})"
            continue # このページの処理を中断

        soup = BeautifulSoup(res.text, 'html.parser')

        title_tag = soup.title
        if title_tag and title_tag.string:
            title_text = title_tag.string.strip() # .string で中身を取得し、strip()で前後の空白削除
        else:
            title_text = 'タイトルなし' # titleタグがない、または中身が空の場合
        
        page_data[current_url] = title_text # 辞書に格納

        links = soup.select('a[href]')

        found_count = 0
        for link in links:
            new_url = urljoin(current_url, link['href'])

            new_url = new_url.split('#')[0]

            parsed_new_url = urlparse(new_url)

            if parsed_new_url.scheme not in ['http', 'https']:
                continue

            if new_url.lower().endswith(EXCLUDED_EXTENSIONS):
                continue

            if parsed_new_url.netloc == base_domain:
                if new_url not in visited_url and new_url not in visit_url_queue:
                    visit_url_queue.append(new_url)
                    found_count += 1


        print(f"Found {found_count} new URLs on {current_url}")

    except Exception as e:
        print(f"Error accessing {current_url}: {e}")

print("\n--- クローリング終了 ---")
print(f"訪問した総ページ数: {len(visited_url)}")

# -----------------------------------
# 課題対応：最終的な辞書を表示
# -----------------------------------
print("\n--- 取得した URL と タイトル の辞書 ---")
# 件数が多いと見にくいため、pprint を使うとより見やすくなります
# from pprint import pprint
# pprint(page_data)
print(page_data)

[1件目] Accessing: https://www.musashino-u.ac.jp
Found 99 new URLs on https://www.musashino-u.ac.jp
[2件目] Accessing: https://www.musashino-u.ac.jp/
Found 0 new URLs on https://www.musashino-u.ac.jp/
[3件目] Accessing: https://www.musashino-u.ac.jp/access.html
Found 8 new URLs on https://www.musashino-u.ac.jp/access.html
[4件目] Accessing: https://www.musashino-u.ac.jp/admission/request.html
Found 31 new URLs on https://www.musashino-u.ac.jp/admission/request.html
[5件目] Accessing: https://www.musashino-u.ac.jp/contact.html
Found 0 new URLs on https://www.musashino-u.ac.jp/contact.html
[6件目] Accessing: https://www.musashino-u.ac.jp/prospective-students.html
Found 8 new URLs on https://www.musashino-u.ac.jp/prospective-students.html
[7件目] Accessing: https://www.musashino-u.ac.jp/students.html
Found 8 new URLs on https://www.musashino-u.ac.jp/students.html
[8件目] Accessing: https://www.musashino-u.ac.jp/alumni.html
Found 2 new URLs on https://www.musashino-u.ac.jp/alumni.html
[9件目] Accessing: htt